# ECO225 Project 2 — Final Report Notebook
**Research Question:** How does economic development (GDP per capita) affect Kiva lending activity across countries, and does institutional quality moderate this relationship?

**Main finding (development story):** Microfinance lending activity is highest in **middle-income** countries (an inverted U-shaped relationship between development and Kiva lending).

**Data:** Country-year panel, 2013–2017.
- Unit: country × year
- Outcome (Y): `log_total_loan_amount`
- Core regressor (X): `log_gdp_pc`
- Institutions (Z): `institutional_pca1` (main), `institutional_index` (alt)
- Controls: `log_population`
- Robustness: `financial_access_index`, `poverty_rate` (high missingness)

> **Note on output tables:** This notebook uses `summary_col` from statsmodels (stable) instead of the `stargazer` package (which can error in some environments).

## 0. Setup
Run this cell once at the start.

In [9]:
import pandas as pd
import numpy as np

import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

from IPython.display import HTML, display

## 1. Load the final panel
This notebook expects your final panel at:

- `outputs/final_panel_country_year.csv`

If your path differs, change `DATA_PATH` below.

In [10]:
DATA_PATH = "outputs/final_panel_country_year.csv"
df = pd.read_csv(DATA_PATH)

print("Rows:", df.shape[0], "Cols:", df.shape[1])
print("Columns:", list(df.columns))
print("Years:", sorted(df["year"].dropna().unique()))
print("Countries:", df["country_code"].nunique() if "country_code" in df.columns else "country_code not found")

Rows: 357 Cols: 12
Columns: ['country_code', 'year', 'n_loans', 'total_loan_amount', 'log_total_loan_amount', 'log_gdp_pc', 'population', 'log_population', 'institutional_pca1', 'institutional_index', 'financial_access_index', 'poverty_rate']
Years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
Countries: 84


## 2. Variables and regression sample
We build a consistent baseline sample with complete cases on the core variables.

In [11]:
# Core variables for baseline regressions
core_vars = [
    "log_total_loan_amount",
    "log_gdp_pc",
    "institutional_pca1",
    "log_population",
    "year",
    "country_code",
]

missing = [c for c in core_vars if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

reg = df[core_vars].dropna().copy()
reg["year"] = reg["year"].astype(int)

print("Baseline sample N (country-year):", reg.shape[0])
print("Countries:", reg["country_code"].nunique(), "| Years:", reg["year"].nunique())

Baseline sample N (country-year): 348
Countries: 81 | Years: 5


## 3. Helper: OLS with robust / clustered standard errors
- `HC1`: heteroskedasticity-robust (default)
- `cluster`: cluster-robust by country (`country_code`)

In [12]:
def fit_ols(formula: str, data: pd.DataFrame, se: str = "HC1", cluster_col: str | None = None):
    m = smf.ols(formula, data=data).fit()
    if se.upper() == "HC1":
        return m.get_robustcov_results(cov_type="HC1")
    if se.lower() == "cluster":
        if cluster_col is None:
            raise ValueError("cluster_col must be provided when se='cluster'")
        return m.get_robustcov_results(cov_type="cluster", groups=data[cluster_col])
    raise ValueError("se must be 'HC1' or 'cluster'")

## 4. Descriptive statistics
Quick summary of the main variables (baseline sample).

In [13]:
desc_vars = ["log_total_loan_amount", "log_gdp_pc", "institutional_pca1", "log_population"]
display(reg[desc_vars].describe().T)

,count,mean,std,min,25%,50%,75%,max
log_total_loan_amount,348.0,13.005601,1.902406,7.150701,11.713253,13.147007,14.554529,16.601689
log_gdp_pc,348.0,7.827715,1.068907,5.538918,7.045118,7.756100,8.615553,10.970810
institutional_pca1,348.0,-0.137251,0.869529,-2.551277,-0.567676,-0.084743,0.395790,1.955123
log_population,348.0,16.672488,1.609349,12.188112,15.668342,16.588435,17.708669,21.066513


## 5. Baseline regressions (Models 1–4)
We estimate:

1) \( \log(Loan_{it}) = \beta_1 \log(GDPpc_{it}) + \varepsilon_{it} \)

2) Add institutions (PCA)

3) Add population control

4) Interaction (mechanism): \( \log(GDPpc) \times \text{Institutions} \)

We report **country-clustered** standard errors for Models (3) and (4), since this is a panel by country.

In [14]:
# Model formulas
f1 = "log_total_loan_amount ~ log_gdp_pc"
f2 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1"
f3 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1 + log_population"
f4 = "log_total_loan_amount ~ log_gdp_pc * institutional_pca1 + log_population"

# Fit models
m1 = fit_ols(f1, reg, se="HC1")
m2 = fit_ols(f2, reg, se="HC1")
m3 = fit_ols(f3, reg, se="cluster", cluster_col="country_code")
m4 = fit_ols(f4, reg, se="cluster", cluster_col="country_code")

tbl1 = summary_col(
    [m1, m2, m3, m4],
    stars=True,
    model_names=["(1)", "(2)", "(3) Cluster", "(4) Cluster"],
    info_dict={
        "N":  lambda x: f"{int(x.nobs)}",
        "R2": lambda x: f"{x.rsquared:.3f}"
    }
)

display(HTML(tbl1.as_html()))

,(1),(2),(3) Cluster,(4) Cluster
Intercept,13.4346***,14.1660***,12.2128***,11.8249***
,(0.7764),(0.9270),(1.9682),(2.1807)
log_gdp_pc,-0.0548,-0.1449,-0.1805,-0.1369
,(0.1000),(0.1184),(0.1580),(0.1548)
institutional_pca1,,0.1893,0.2512,1.9981
,,(0.1450),(0.1885),(1.3544)
log_population,,,0.1344,0.1443
,,,(0.0967),(0.1027)
log_gdp_pc:institutional_pca1,,,,-0.2267
,,,,(0.1806)


### Interpretation guide (baseline)
- `log_gdp_pc` is the elasticity of total lending with respect to GDP per capita (in logs).
- `institutional_pca1` captures institutional quality (higher = better).
- `log_population` controls for country size.
- The interaction term tests whether the GDP–lending relationship changes with institutional quality.

## 6. Core result: Nonlinearity (Quadratic GDP) — “middle-income peak”
To test the development story, we estimate a quadratic specification:

\[
\log(Loan_{it}) = \beta_1 \log(GDPpc_{it}) + \beta_2 \log(GDPpc_{it})^2 + \gamma Inst_{it} + \delta \log(Pop_{it}) + \varepsilon_{it}
\]

An inverted U-shape corresponds to **\(\beta_1 > 0\)** and **\(\beta_2 < 0\)**.

In [15]:
reg = reg.copy()
reg["log_gdp_pc_sq"] = reg["log_gdp_pc"] ** 2

f5 = "log_total_loan_amount ~ log_gdp_pc + log_gdp_pc_sq + institutional_pca1 + log_population"
m5 = fit_ols(f5, reg, se="cluster", cluster_col="country_code")

tbl2a = summary_col(
    [m5],
    stars=True,
    model_names=["(5) Quad GDP (Cluster)"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)
display(HTML(tbl2a.as_html()))

,(5) Quad GDP (Cluster)
Intercept,2.0449
,(7.1085)
log_gdp_pc,2.3154
,(1.8345)
log_gdp_pc_sq,-0.1553
,(0.1161)
institutional_pca1,0.2739
,(0.1855)
log_population,0.1538
,(0.1003)


### Turning point (optional)
For an inverted U-shape, the turning point in log GDP per capita is:

\[
\log(GDPpc)^* = -\frac{\beta_1}{2\beta_2}
\]

We compute it below (if \(\beta_2\neq 0\)).

In [16]:
coef_names = m5.model.exog_names
coef_dict = dict(zip(coef_names, m5.params))

b1 = coef_dict.get("log_gdp_pc", np.nan)
b2 = coef_dict.get("log_gdp_pc_sq", np.nan)

if np.isfinite(b1) and np.isfinite(b2) and b2 != 0:
    turning_log = -b1/(2*b2)
    turning_level = np.exp(turning_log)

    print("Turning point log(GDPpc):", turning_log)
    print("Turning point GDPpc (level units):", turning_level)
else:
    print("Could not compute turning point.")

Turning point log(GDPpc): 7.456464177281394
Turning point GDPpc (level units): 1731.016655944208


## 7. Year fixed effects
Kiva platform activity may change over time due to global expansion or macro shocks. We add year fixed effects:

\[
\log(Loan_{it}) = \beta \log(GDPpc_{it}) + \gamma Inst_{it} + \delta \log(Pop_{it}) + \lambda_t + \varepsilon_{it}
\]

In [17]:
f6 = "log_total_loan_amount ~ log_gdp_pc + institutional_pca1 + log_population + C(year)"
m6 = fit_ols(f6, reg, se="cluster", cluster_col="country_code")

tbl2b = summary_col(
    [m6],
    stars=True,
    model_names=["(6) + Year FE (Cluster)"],
    info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
)
display(HTML(tbl2b.as_html()))

,(6) + Year FE (Cluster)
Intercept,9.8013***
,(2.1711)
C(year)[T.2014],2.6640***
,(0.2175)
C(year)[T.2015],2.6852***
,(0.2039)
C(year)[T.2016],2.8683***
,(0.1998)
C(year)[T.2017],2.2795***
,(0.1964)


## 8. Alternative institutional measure (robustness): `institutional_index`
We re-estimate the institution and interaction models using the alternative index.

In [18]:
need_cols = ["log_total_loan_amount", "log_gdp_pc", "institutional_index", "log_population", "year", "country_code"]
if all(c in df.columns for c in need_cols):
    reg2 = df[need_cols].dropna().copy()
    reg2["year"] = reg2["year"].astype(int)

    f7 = "log_total_loan_amount ~ log_gdp_pc + institutional_index + log_population"
    f8 = "log_total_loan_amount ~ log_gdp_pc * institutional_index + log_population"

    m7 = fit_ols(f7, reg2, se="cluster", cluster_col="country_code")
    m8 = fit_ols(f8, reg2, se="cluster", cluster_col="country_code")

    tbl2c = summary_col(
        [m7, m8],
        stars=True,
        model_names=["(7) inst_index", "(8) inst_index × GDP"],
        info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
    )
    display(HTML(tbl2c.as_html()))
else:
    print("institutional_index block skipped (missing columns):", [c for c in need_cols if c not in df.columns])

,(7) inst_index,(8) inst_index × GDP
Intercept,12.3339***,11.8857***
,(1.9692),(2.1656)
log_gdp_pc,-0.1851,-0.1379
,(0.1603),(0.1565)
institutional_index,0.2746,2.1864
,(0.2086),(1.4663)
log_population,0.1292,0.1414
,(0.0951),(0.1011)
log_gdp_pc:institutional_index,,-0.2467
,,(0.1952)


## 9. Alternative “financial development” proxy (robustness): `financial_access_index`
This variable has more missingness, so sample size may drop.

In [19]:
need_cols = ["log_total_loan_amount", "log_gdp_pc", "financial_access_index", "log_population", "year", "country_code"]
if all(c in df.columns for c in need_cols):
    reg3 = df[need_cols].dropna().copy()
    reg3["year"] = reg3["year"].astype(int)

    f9  = "log_total_loan_amount ~ log_gdp_pc + financial_access_index + log_population"
    f10 = "log_total_loan_amount ~ log_gdp_pc * financial_access_index + log_population"

    m9  = fit_ols(f9, reg3, se="cluster", cluster_col="country_code")
    m10 = fit_ols(f10, reg3, se="cluster", cluster_col="country_code")

    tbl2d = summary_col(
        [m9, m10],
        stars=True,
        model_names=["(9) fin_access", "(10) fin_access × GDP"],
        info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
    )
    display(HTML(tbl2d.as_html()))
else:
    print("financial_access_index block skipped (missing columns):", [c for c in need_cols if c not in df.columns])

,(9) fin_access,(10) fin_access × GDP
Intercept,13.1586***,12.9591***
,(2.5195),(2.5858)
log_gdp_pc,-0.2083,-0.1709
,(0.2275),(0.2509)
financial_access_index,0.1611,-0.3462
,(0.1991),(1.0854)
log_population,0.0918,0.0843
,(0.0923),(0.0953)
log_gdp_pc:financial_access_index,,0.0497
,,(0.0986)


## 10. Poverty rate robustness (optional; high missingness)
Because `poverty_rate` is missing for many country-years, we treat this as a robustness check only.

In [20]:
need_cols = ["log_total_loan_amount", "log_gdp_pc", "poverty_rate", "log_population", "year", "country_code"]
if all(c in df.columns for c in need_cols):
    reg4 = df[need_cols].dropna().copy()
    reg4["year"] = reg4["year"].astype(int)

    f11 = "log_total_loan_amount ~ log_gdp_pc + poverty_rate + log_population"
    m11 = fit_ols(f11, reg4, se="cluster", cluster_col="country_code")

    tbl2e = summary_col(
        [m11],
        stars=True,
        model_names=["(11) + Poverty (Cluster)"],
        info_dict={"N": lambda x: f"{int(x.nobs)}", "R2": lambda x: f"{x.rsquared:.3f}"}
    )
    display(HTML(tbl2e.as_html()))
    print("Poverty robustness sample N:", reg4.shape[0])
else:
    print("poverty_rate block skipped (missing columns):", [c for c in need_cols if c not in df.columns])

,(11) + Poverty (Cluster)
Intercept,15.5426***
,(3.8779)
log_gdp_pc,-0.3107
,(0.3492)
poverty_rate,-0.0077
,(0.0137)
log_population,0.0184
,(0.1539)
R-squared,0.0164
R-squared Adj.,-0.0037


Poverty robustness sample N: 151


## 11. Results write-up (copy-ready)
Use this text in your report (edit numbers if you want to reference exact coefficients).

**Development story (main message).**  
Across specifications, the relationship between development and Kiva lending is not purely monotonic. The quadratic specification yields an inverted U-shaped pattern: lending rises with GDP per capita at low levels of development but declines at higher income levels. This implies that microfinance activity is highest in middle-income countries rather than in the poorest or richest economies.

**Institutions as a moderator.**  
Interaction models suggest institutional quality meaningfully shapes the development gradient. In higher-quality institutional environments, the decline in microfinance activity at higher income levels is more pronounced, consistent with faster substitution toward formal financial markets as countries develop.

**Time effects.**  
Including year fixed effects substantially increases explanatory power, indicating that global time variation (platform growth and common shocks) is an important driver of lending volumes.

## 12. Conclusion (copy-ready)

This project provides evidence that Kiva microfinance lending follows a development “life-cycle.” Rather than being concentrated in the very poorest countries, lending activity is strongest in middle-income economies, consistent with operational feasibility constraints at very low development levels and substitution by formal finance at higher income levels. Institutional quality further moderates the development effect, suggesting that stronger institutions accelerate the transition away from microfinance as economies grow.